### Notebook 11 — Real-Data Experiment (A): ABIDE fMRI
### Do LCT and Fisher-z agree on clean, near-Gaussian fMRI?

In [1]:
# --- Cell 1: imports + fetch (NYU, Craddock-200) ---
import sys, numpy as np, pandas as pd
from pathlib import Path
from nilearn import datasets
sys.path.insert(0, '..')   # so we can import src/

abide = datasets.fetch_abide_pcp(
    derivatives=['rois_cc200'],
    pipeline='cpac',
    band_pass_filtering=True,
    global_signal_regression=False,
    quality_checked=True,
    SITE_ID='NYU',
)
ts_list = abide['rois_cc200']
pheno   = abide['phenotypic']
print("subjects fetched:", len(ts_list))
print("example shape (timepoints x ROIs):", ts_list[0].shape)

[fetch_abide_pcp] Dataset directory found: C:\Users\hirshikesh\nilearn_data\ABIDE_pcp
subjects fetched: 172
example shape (timepoints x ROIs): (176, 200)


In [2]:
# --- Cell 2: split into groups ---
dx = pheno['DX_GROUP']                 # 1 = ASD, 2 = control
asd_idx = np.where(dx == 1)[0]
ctl_idx = np.where(dx == 2)[0]
print(f"ASD subjects:     {len(asd_idx)}")
print(f"Control subjects: {len(ctl_idx)}")
print("ROI counts:", set(t.shape[1] for t in ts_list))
print("timepoint range:", min(t.shape[0] for t in ts_list), "-", max(t.shape[0] for t in ts_list))

ASD subjects:     74
Control subjects: 98
ROI counts: {200}
timepoint range: 176 - 176


In [3]:
# --- Cell 3: measure kappa (the bridge to simulation) ---
from src.LCT import _kappa_hat, _zscore_columns

def clean_subject(X):
    """True if usable: enough timepoints, no flat (zero-variance) ROI."""
    return X.shape[0] >= 30 and not np.any(X.std(axis=0) < 1e-8)

def group_kappa(indices):
    ks = [_kappa_hat(_zscore_columns(ts_list[i])) for i in indices if clean_subject(ts_list[i])]
    return np.array(ks)

k_asd, k_ctl = group_kappa(asd_idx), group_kappa(ctl_idx)
print(f"kappa_hat ASD:     mean {k_asd.mean():.3f}  (Gaussian = 1.0)")
print(f"kappa_hat Control: mean {k_ctl.mean():.3f}")
print(f"fraction with kappa > 1.1: {np.mean(np.concatenate([k_asd,k_ctl]) > 1.1):.2f}")

kappa_hat ASD:     mean 1.016  (Gaussian = 1.0)
kappa_hat Control: mean 1.013
fraction with kappa > 1.1: 0.05


In [4]:
# --- Cell 4: group-representative correlation matrices, honest n ---
# Average per-subject correlation matrices. Effective n = number of subjects
# (the independent units), NOT stacked timepoints. Skip subjects with a flat ROI.
def group_corr_matrix(indices):
    mats, skipped = [], 0
    for i in indices:
        X = ts_list[i]
        if not clean_subject(X):
            skipped += 1; continue
        mats.append(np.corrcoef(X, rowvar=False))
    print(f"  used {len(mats)}, skipped {skipped}")
    return np.mean(mats, axis=0), len(mats)

print("ASD:");     R_asd, n_asd = group_corr_matrix(asd_idx)
print("Control:"); R_ctl, n_ctl = group_corr_matrix(ctl_idx)
p = 200
iu, ju = np.triu_indices(p, 1)
print(f"\nHonest n: ASD={n_asd}, Control={n_ctl}")
print(f"mean |corr| ASD={np.abs(R_asd[iu,ju]).mean():.3f}, CTL={np.abs(R_ctl[iu,ju]).mean():.3f}")
print("any NaN?", np.isnan(R_asd).any(), np.isnan(R_ctl).any())

ASD:
  used 74, skipped 0
Control:
  used 96, skipped 2

Honest n: ASD=74, Control=96
mean |corr| ASD=0.231, CTL=0.231
any NaN? False False


In [5]:
# --- Cell 5: Fisher-z + BH/BY at honest n ---
from src.FisherBaselines import two_group_z_stat, pvals_from_Z, bh_threshold, by_threshold

def count(m):
    return int(m[iu,ju].sum()) if m.ndim == 2 else int(m.sum())

Z  = two_group_z_stat(R_asd, R_ctl, n_asd, n_ctl)
pv = pvals_from_Z(Z)[iu, ju]
sel_bh = bh_threshold(pv, 0.05)
sel_by = by_threshold(pv, 0.05)

print(f"Discoveries at alpha=0.05 (of {len(iu):,} edges):")
print(f"  Fisher-z + BH : {count(sel_bh)}")
print(f"  Fisher-z + BY : {count(sel_by)}")

Discoveries at alpha=0.05 (of 19,900 edges):
  Fisher-z + BH : 0
  Fisher-z + BY : 0


In [6]:
# --- Cell 6: LCT at honest n ---
# LCT's data-array entry point would compute n from stacked timepoints (inflated).
# Instead we compute the LCT edge statistic directly from the two group
# correlation matrices and the honest subject counts, using the same
# Cai-Liu Eq.(4) variance the simulations use.
from src.LCT import _kappa_hat, _zscore_columns, _rho_tilde_sq, lct_threshold_normal
from src.LCTB_v2 import lct_threshold_bootstrap
from scipy.stats import norm

# group-level kappa (mean of clean per-subject kappas)
kappa1 = float(np.mean(k_asd))
kappa2 = float(np.mean(k_ctl))

# thresholded rho^2 (shared denominator term), then the Eq.(4) T statistic
rt2 = _rho_tilde_sq(R_asd, R_ctl, kappa1, kappa2, n_asd, n_ctl, p)
shared = (1.0 - rt2) ** 2
V1 = (kappa1 / n_asd) * shared
V2 = (kappa2 / n_ctl) * shared
np.fill_diagonal(V1, 0.0); np.fill_diagonal(V2, 0.0)
denom = np.sqrt(V1 + V2)
with np.errstate(divide="ignore", invalid="ignore"):
    T = (R_asd - R_ctl) / denom
np.fill_diagonal(T, 0.0)
T = np.nan_to_num(T, nan=0.0, posinf=0.0, neginf=0.0)

# LCT-N threshold on this T
_, mask_lctn = lct_threshold_normal(T, alpha=0.05)

print(f"group kappa: ASD={kappa1:.3f}, Control={kappa2:.3f}")
print(f"LCT-N discoveries: {count(mask_lctn)}")

group kappa: ASD=1.016, Control=1.013
LCT-N discoveries: 0


In [7]:
# --- Cell 7: agreement between LCT-N and Fisher-z+BH ---
lctn_set = set(np.where(mask_lctn[iu,ju] if mask_lctn.ndim==2 else mask_lctn)[0])
bh_set   = set(np.where(sel_bh[iu,ju]   if sel_bh.ndim==2   else sel_bh)[0])
inter, union = len(lctn_set & bh_set), len(lctn_set | bh_set)

print(f"LCT-N discoveries: {len(lctn_set)}")
print(f"BH discoveries:    {len(bh_set)}")
print(f"overlap (Jaccard): {inter/max(union,1):.3f}")
print(f"only LCT-N: {len(lctn_set - bh_set)}   only BH: {len(bh_set - lctn_set)}")

LCT-N discoveries: 0
BH discoveries:    0
overlap (Jaccard): 0.000
only LCT-N: 0   only BH: 0


In [8]:
# --- Cell 8: summary for the paper ---
print("=== ABIDE (NYU, cc200) real-data summary ===")
print(f"Subjects: {n_asd} ASD, {n_ctl} control")
print(f"Kurtosis: ASD kappa={kappa1:.3f}, Control kappa={kappa2:.3f} (near-Gaussian)")
print(f"Discoveries @ alpha=0.05 of {len(iu):,} edges:")
print(f"   BH={count(sel_bh)}  BY={count(sel_by)}  LCT-N={count(mask_lctn)}")
print(f"LCT-N vs BH overlap (Jaccard): {inter/max(union,1):.3f}")
print("\nInterpretation: near-Gaussian kurtosis -> methods largely agree,")
print("consistent with the Gaussian regime of the simulation study.")

=== ABIDE (NYU, cc200) real-data summary ===
Subjects: 74 ASD, 96 control
Kurtosis: ASD kappa=1.016, Control kappa=1.013 (near-Gaussian)
Discoveries @ alpha=0.05 of 19,900 edges:
   BH=0  BY=0  LCT-N=0
LCT-N vs BH overlap (Jaccard): 0.000

Interpretation: near-Gaussian kurtosis -> methods largely agree,
consistent with the Gaussian regime of the simulation study.


In [9]:
# sanity: what do the raw p-values look like? and the effect sizes?
print("p-value distribution:")
print(f"  min p: {pv.min():.2e}")
print(f"  # p < 0.05 (uncorrected): {(pv < 0.05).sum()}")
print(f"  # p < 0.001: {(pv < 0.001).sum()}")

# effect sizes: how big are the correlation differences?
diff = np.abs(R_asd - R_ctl)[iu, ju]
print(f"\ncorrelation differences |R_asd - R_ctl|:")
print(f"  mean {diff.mean():.3f}, median {np.median(diff):.3f}, "
      f"max {diff.max():.3f}, 99th pct {np.percentile(diff,99):.3f}")

# what would BH need? smallest p vs BH threshold
print(f"\nBH threshold at 0.05: p must be < {0.05 * (np.arange(1,len(pv)+1)/len(pv))[np.argsort(pv)][0]:.2e} for the smallest")

p-value distribution:
  min p: 2.35e-01
  # p < 0.05 (uncorrected): 0
  # p < 0.001: 0

correlation differences |R_asd - R_ctl|:
  mean 0.026, median 0.022, max 0.149, 99th pct 0.088

BH threshold at 0.05: p must be < 2.25e-03 for the smallest
